In [1]:
import pandas as pd
import numpy as np
from typing import Union, Type
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec, FastText
from mittens import GloVe
from scipy.sparse import lil_matrix
import re

# Import utils
from utils import get_vocab, build_vocab_dict, tokenize_corpus, save_embeddings_vec

# Data preparation and cleaning

pairs of english to french sentences from here: https://tatoeba.org/fr/downloads

In [2]:
data = pd.read_csv("./data/french_english.tsv", sep='\t')
data.columns = ["id_en", "en", "id_fr", "fr"]
data.head()

,id_en,en,id_fr,fr
0,1276,Let's try something.,456963,Tentons quelque chose !
1,1277,I have to go to sleep.,373908,Je dois aller dormir.
2,1280,Today is June 18th and it is Muiriel's birthday!,3095,Aujourd'hui nous sommes le 18 juin et c'est l'...
3,1280,Today is June 18th and it is Muiriel's birthday!,696081,"Aujourd'hui c'est le 18 juin, et c'est l'anniv..."
4,1282,Muiriel is 20 now.,3097,Muiriel a 20 ans maintenant.


In [3]:
data.shape

(429470, 4)

The data is very big and would yield a vocabulary of size 46732 for french and 33044 for english, and thus a file of size 8Gb for french and 4Gb for english for OHE embeddings! Therefore, we choose to only sample 20k random sentences of data instead of the full 429470 sentences present in the original text corpus.

In [4]:
data = data.sample(n=20000, random_state=42).reset_index(drop=True)
data.shape

(20000, 4)

In [5]:
pattern_fr = re.compile(r"[a-zA-ZÀ-ÿ]+")
pattern_en = re.compile(r"[a-z]+")

In [8]:
# full vocabs for neural embeddings
french_vocab = get_vocab(data["fr"], pattern_fr)
english_vocab = get_vocab(data["en"], pattern_en)
word_to_idx_fr = build_vocab_dict(french_vocab)
word_to_idx_en = build_vocab_dict(english_vocab)

# reduced vocabs for OHE and TF-IDF encodings
french_vocab_small = get_vocab(data["fr"], pattern_fr, max_vocab=1700)
english_vocab_small = get_vocab(data["en"], pattern_en, max_vocab=1700)
word_to_idx_fr_small = build_vocab_dict(french_vocab_small)
word_to_idx_en_small = build_vocab_dict(english_vocab_small)

In [9]:
print(f"French vocab: {len(french_vocab)} words")
print(f"English vocab: {len(english_vocab)} words")
print(f"Small French vocab: {len(french_vocab_small)} words")
print(f"Small English vocab: {len(english_vocab_small)} words")
print(english_vocab[-20:])
print(french_vocab[-20:])

French vocab: 12094 words
English vocab: 9117 words
Small French vocab: 1700 words
Small English vocab: 1700 words
['youth', 'yuck', 'yummy', 'zailaiba', 'zamenhof', 'zap', 'zealand', 'zealous', 'zebras', 'zero', 'zinc', 'zip', 'ziri', 'zohran', 'zombies', 'zoo', 'zoroastrianism', 'zucchini', 'zucchinis', 'zurich']
['évites', 'évité', 'évitées', 'évolue', 'évolution', 'évoquant', 'évoque', 'évoqué', 'évènements', 'événement', 'événements', 'êtes', 'être', 'êtres', 'île', 'îles', 'ô', 'ôta', 'ôtez', 'ôtée']


We can see that the new reduced data yielded vocabularies with a size around the 10k mark as oposed to the 40k from the full data.\
Plus the small vocabulary that we will use for OHE and TF-IDF encoding will consist of 1700 tokens (to match the dimension of the full matrix of embeddings from the neural approach: 10k vocab * 300 embedding dim = 3M ~= 1700^2)

In [10]:
french_sentences = tokenize_corpus(data["fr"], pattern_fr)
english_sentences = tokenize_corpus(data["en"], pattern_en)
print(tokenize_corpus(data["fr"], pattern_fr)[:5])

[['je', 'sais', 'que', 'tom', 'est', 'en', 'bas'], ['ziri', 'a', 'mangé', 'des', 'pommes', 'de', 'terre'], ['père', 'arrose', 'les', 'fleurs'], ['maintenant', 'calme', 'toi'], ['nous', 'avons', 'dû', 'céder', 'à', 'leur', 'demande']]


# Word Embeddings Creation

#### One Hot Encoding word vectors

In [12]:
def create_onehot_embeddings(vocab: list[str], output_file: str):
    """Create one-hot encoded word vectors."""
        
    vocab_size = len(vocab)
    onehot_matrix = np.eye(vocab_size, dtype=np.float32)
    
    print(f"saving to {output_file}")    
    save_embeddings_vec(vocab, onehot_matrix, output_file)
    return onehot_matrix

In [13]:
create_onehot_embeddings(french_vocab_small, "./embeddings/ohe/french_onehot.vec")
create_onehot_embeddings(english_vocab_small, "./embeddings/ohe/english_onehot.vec")

saving to ./embeddings/ohe/french_onehot.vec
saving to ./embeddings/ohe/english_onehot.vec


array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(1700, 1700), dtype=float32)

#### TF-IDF word vectors

Using tf-idf scores as token embeddings is clearly not the best approach out there, and using them when the number of documents is very large like the text corpus we're using in this notebook is even less ideal (since we end up with embeddings with the same dimension as the number of documents).
Therefore, the approach we choose to take is to slightly enhance the one-hot encoding representation by weighing each vector by the mean of its TF-IDF score across all documents.

In [14]:
def create_tfidf_embeddings(sentences: list[list[str]], vocab: list[str], output_file: str):
    """Create TF-IDF word vectors (one-hot scaled by TF-IDF scores)"""
    
    # Join sentences for TfidfVectorizer
    texts = [' '.join(sent) for sent in sentences]
    
    vectorizer = TfidfVectorizer(vocabulary=vocab)
    tfidf_matrix = vectorizer.fit_transform(texts)
    
    # Get TF-IDF score for each word (average across all documents)
    tfidf_scores = np.array(tfidf_matrix.mean(axis=0)).flatten()
    
    # Create embeddings: eye(vocab_size) * tfidf_scores
    vocab_size = len(vocab)
    word_vectors = np.eye(vocab_size, dtype=np.float32) * tfidf_scores[:, np.newaxis]
    
    print(f"saving to {output_file}")
    save_embeddings_vec(vocab, word_vectors, output_file)
    return word_vectors

In [15]:
create_tfidf_embeddings(french_sentences, french_vocab_small, "./embeddings/tfidf/french_tfidf.vec")
create_tfidf_embeddings(english_sentences, english_vocab_small, "./embeddings/tfidf/english_tfidf.vec")

saving to ./embeddings/tfidf/french_tfidf.vec
saving to ./embeddings/tfidf/english_tfidf.vec


array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.00017023, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.00100803, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.0015196 , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.00030196,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.00306924]], shape=(1700, 1700))

### Neural embeddings

In [16]:
def train_neural_embeddings(sentences: list[list[str]], model_class: Type[Union[Word2Vec, FastText]], vec_file: str, model_file: str, model_name: str, vector_size: int = 300):
    """Train Word2Vec or FastText embeddings and save both .vec and full model"""
    print(f"training {model_name}")
    model = model_class(sentences=sentences, vector_size=vector_size, min_count=5, seed=42)
    
    print(f"saving to {vec_file}")
    model.wv.save_word2vec_format(vec_file, binary=False)
    
    print(f"saving full model to {model_file}")
    model.save(model_file)
    
    return model

#### wor2vec embeddings

In [17]:
train_neural_embeddings(french_sentences, Word2Vec, "./embeddings/w2v/french_word2vec.vec", "./embeddings/w2v/french_word2vec.model", "Word2Vec")
train_neural_embeddings(english_sentences, Word2Vec, "./embeddings/w2v/english_word2vec.vec", "./embeddings/w2v/english_word2vec.model", "Word2Vec")

training Word2Vec
saving to ./embeddings/w2v/french_word2vec.vec
training Word2Vec
saving to ./embeddings/w2v/english_word2vec.vec


#### FastText embeddings

In [19]:
train_neural_embeddings(french_sentences, FastText, "./embeddings/ft/french_fasttext.vec", "./embeddings/ft/french_fasttext.model", "FastText")
train_neural_embeddings(english_sentences, FastText, "./embeddings/ft/english_fasttext.vec", "./embeddings/ft/english_fasttext.model", "FastText")

training FastText
saving to ./embeddings/ft/french_fasttext.vec
training FastText
saving to ./embeddings/ft/english_fasttext.vec


#### GloVe embeddings

code for co-occurence matrix building comes from : https://www.foldl.me/2014/glove-python/ + LLM usage

In [20]:

from scipy.sparse import lil_matrix

def create_glove_embeddings(sentences, vocab, word_to_idx, output_file, vector_size=300):
    vocab_size = len(vocab)
    
    # Use SPARSE matrix for co-occurrence (saves memory)
    cooccur = lil_matrix((vocab_size, vocab_size), dtype=np.float64)
    window = 3
    
    for sentence in sentences:
        word_ids = [word_to_idx[word] for word in sentence if word in word_to_idx]
        
        for i, word_id in enumerate(word_ids):
            context_start = max(0, i - window)
            
            for j in range(context_start, i):
                context_id = word_ids[j]
                distance = i - j
                increment = 1.0 / distance
                
                cooccur[word_id, context_id] += increment
                cooccur[context_id, word_id] += increment
    
    # Convert to dense for GloVe training
    cooccur_dense = cooccur.toarray()
    
    glove_model = GloVe(n=vector_size, max_iter=100)
    
    print(f"training GloVe")
    embeddings = glove_model.fit(cooccur_dense)  # Dense output
    
    print(f"saving to {output_file}")
    save_embeddings_vec(vocab, embeddings, output_file)
    return embeddings


In [21]:
create_glove_embeddings(french_sentences, french_vocab, word_to_idx_fr, "./embeddings/glv/french_glove.vec")
create_glove_embeddings(english_sentences, english_vocab, word_to_idx_en, "./embeddings/glv/english_glove.vec")

Iteration 100: error 105.0185

array([[ 2.91562241e-02,  3.70651336e-01,  3.43421645e-01, ...,
         3.47536703e-01,  2.65871938e-01, -5.32894913e-02],
       [-2.90199318e-03, -1.69166233e-02, -4.18376613e-02, ...,
        -6.17813325e-04,  2.82265320e-02, -2.72181937e-02],
       [ 5.39536424e-02, -4.80038444e-02,  6.36389636e-05, ...,
         5.43021286e-03, -4.22184987e-02,  3.02625218e-03],
       ...,
       [-7.45531157e-03, -6.66393695e-03,  9.82754217e-03, ...,
         6.84689114e-03,  3.41026867e-02,  1.35272033e-03],
       [ 2.44383271e-02,  8.85731903e-03,  8.86496212e-03, ...,
        -2.72046087e-02,  3.58414686e-02, -2.27543902e-03],
       [-6.50197774e-03,  1.12959435e-02, -2.22355457e-02, ...,
        -1.77086486e-02,  3.61737685e-02,  1.62334153e-02]],
      shape=(9117, 300))